# 1.5 AirSim Multi-Drone Control

AirSim's multi-drone feature supports simultaneous simulation and control of multiple drones, suitable for swarm tasks, cooperative navigation, and other complex scenarios. Through configuration files (e.g., `settings.json`), you can define drone initial positions, types, and communication parameters, and use the Python API or ROS for distributed control. For example, users can create multiple `MultirotorClient` instances in a loop, connecting to different ports to achieve synchronized multi-drone takeoff, path planning, and sensor data sharing.

This feature also supports dynamic environment interaction, such as weather simulation, lighting changes, and custom terrain, enabling verification of autonomous navigation in GPS-denied environments. In advanced applications, AirSim combined with PX4 and ROS frameworks can build distributed simulation platforms supporting multi-drone cooperative obstacle avoidance, task allocation, and other algorithm testing, providing efficient verification tools for logistics delivery, search and rescue, and similar scenarios.

1. **Cooperative Tasks**  
   Implement multi-drone communication via ROS or custom protocols, such as task allocation or cooperative obstacle avoidance.  
2. **Distributed Simulation**  
   Run multiple AirSim instances on multiple machines to simulate large-scale swarm scenarios.  
3. **Visual Debugging**  
   Use RViz to visualize drone states and sensor data.

AirSim typically uses configuration files to set up multiple drones. You can also dynamically add them via Python code, but here we focus on the configuration file approach:

### **1. Configuration File Setup**
1. **Create `settings.json` file**  
   Create a `settings.json` file in the `~/.airsim/` directory, configuring parameters for 3 drones (name, position, type, etc.):  
   ```json
   {
       "SettingsVersion": 1.2,
       "SimMode": "Multirotor",
       "Vehicles": {
           "UAV1": {
               "VehicleType": "SimpleFlight",
               "X": 0,
               "Y": 0,
               "Z": -2,
               "Yaw": 0
           },
           "UAV2": {
               "VehicleType": "SimpleFlight",
               "X": 5,
               "Y": 0,
               "Z": -2,
               "Yaw": 90
           },
           "UAV3": {
               "VehicleType": "SimpleFlight",
               "X": -5,
               "Y": 0,
               "Z": -2,
               "Yaw": 180
           }
       }
   }
   ```  
   - `X/Y/Z` defines the drone's initial position (unit: meters)  
   - `Yaw` defines the initial heading (unit: degrees)

### **2. Python API Control**
1. **Connecting to multiple drones**  
   Simply use different vehicle names:
   ```python
    client = airsim.MultirotorClient()
    plane_name = "UAV3"  # Different drone name
    client.enableApiControl(True, vehicle_name=plane_name)  # All function calls are similar, just add the vehicle_name parameter
    client.armDisarm(True, vehicle_name=plane_name)
    client.takeoffAsync(vehicle_name=plane_name).join()

   ```



With the steps above, you can quickly set up a 3-drone simulation environment in AirSim and control them flexibly through Python.

After copying settings.json to the user directory, you can proceed with the following operations.

## Basic Multi-Drone Control

In [ ]:
import sys
sys.path.append('..')
import airsim
import math
import numpy as np
import time

In [ ]:
client = airsim.MultirotorClient()  # Connect to the AirSim simulator
for i in range(3):
    name = "UAV"+str(i+1)
    client.enableApiControl(True, name)     # Get control
    client.armDisarm(True, name)            # Arm (propellers start spinning)
    if i != 2:                              # Takeoff
        client.takeoffAsync(vehicle_name=name)
    else:
        client.takeoffAsync(vehicle_name=name).join()


for i in range(3):                          # All drones fly to the same altitude
    name = "UAV" + str(i + 1)
    if i != 2:
        client.moveToZAsync(-3, 1, vehicle_name=name)
    else:
        client.moveToZAsync(-3, 1, vehicle_name=name).join()


In [ ]:
# 1. Install nest_asyncio
!pip install nest_asyncio

# 2. Add these two lines at the beginning of your code
import nest_asyncio
nest_asyncio.apply()

In [ ]:
origin_x = [0, 2, 4]       # Drone initial positions
origin_y = [0, 0, 0]

def get_UAV_pos(client, vehicle_name="SimpleFlight"):
    global origin_x
    global origin_y
    state = client.simGetGroundTruthKinematics(vehicle_name=vehicle_name)
    x = state.position.x_val
    y = state.position.y_val
    i = int(vehicle_name[3])
    x += origin_x[i - 1]
    y += origin_y[i - 1]
    pos = np.array([[x], [y]])
    return pos

# Parameter settings
v_max = 2     # Maximum drone flight speed
r_max = 20    # Neighbor selection radius
k_sep = 7     # Control algorithm coefficients
k_coh = 1
k_mig = 1
pos_mig = np.array([[25], [0]])   # Target position
v_cmd = np.zeros([2, 9])

for t in range(500):
    for i in range(3):   # Calculate velocity command for each drone
        name_i = "UAV"+str(i+1)
        pos_i = get_UAV_pos(client, vehicle_name=name_i)
        r_mig = pos_mig - pos_i
        v_mig = k_mig * r_mig / np.linalg.norm(r_mig)
        v_sep = np.zeros([2, 1])
        v_coh = np.zeros([2, 1])
        N_i = 0
        for j in range(3):
            if j != i:
                N_i += 1
                name_j = "UAV"+str(j+1)
                pos_j = get_UAV_pos(client, vehicle_name=name_j)
                if np.linalg.norm(pos_j - pos_i) < r_max:
                    r_ij = pos_j - pos_i
                    v_sep += -k_sep * r_ij / np.linalg.norm(r_ij)
                    v_coh += k_coh * r_ij
                    
        v_sep = v_sep / N_i
        v_coh = v_coh / N_i
        v_cmd[:, i:i+1] = v_sep + v_coh + v_mig

    for i in range(3):   # Execute velocity command for each drone
        name_i = "UAV"+str(i+1)
        client.moveByVelocityZAsync(v_cmd[0, i], v_cmd[1, i], -3, 0.1, vehicle_name=name_i)
    time.sleep(0.01)

In [ ]:
# Loop finished
client.simPause(False)
for i in range(3):
    name = "UAV"+str(i+1)
    if i != 2:                                              # Land
        client.landAsync(vehicle_name=name)
    else:
        client.landAsync(vehicle_name=name).join()

for i in range(3):
    name = "UAV" + str(i + 1)
    client.armDisarm(False, vehicle_name=name)              # Disarm
    client.enableApiControl(False, vehicle_name=name)       # Release control

## Multithreaded Execution
The previous section demonstrated running in a single process using .join() to wait. Here we use multithreading, where each client is responsible for one drone.

The key point to note is that each thread must create its own client - threads cannot share a single client.

Each drone's coordinate system is its own body frame (NED). For unified tasks, you typically use one drone's coordinate system as the reference and perform coordinate transformations.
<img src="img/s1-5-1.png" width='600px' />



NED (North-East-Down) is the core reference frame in drone navigation and control. Its characteristics and applications are as follows:

### **1. NED Definition and Structure**
The NED coordinate system is a **local Cartesian coordinate system** with its origin typically set at the drone's takeoff point or mission start position:

- **X-axis (N-axis)**: Points toward geographic north;

- **Y-axis (E-axis)**: Points toward geographic east;

- **Z-axis (D-axis)**: Points vertically downward toward the earth's center, following the right-hand rule.

Unlike the body frame (Body Frame), the NED coordinate system is a **navigation frame fixed to the ground or inertial space**, used for describing global position and motion state. The body frame is fixed to the drone itself, with its X-axis pointing toward the nose, Y-axis pointing right, and Z-axis pointing down or up (depending on the specific definition).

---

### **2. Core Functions and Applications**
#### **(1) Navigation and Control**
- **Position and Velocity Reference**: The NED frame converts GPS WGS-84 latitude/longitude to local planar coordinates, simplifying path planning and altitude control. For example, a drone's position offset (e.g., 10m north, 5m east) can be directly expressed in NED coordinates.

- **Sensor Data Fusion**: IMU-measured acceleration, gyroscope angular velocity, and other body-frame data must be mapped to the NED frame through coordinate transformation matrices for navigation computation.

#### **(2) Flight Controller Compatibility**
In flight control systems like PX4, the NED coordinate system is the default navigation reference frame. The drone's initial position (0,0,0) corresponds to the takeoff point, and all control commands (target position, velocity) are generated based on this coordinate system.

---

### **3. Relationship with Other Coordinate Systems**
- **GPS (WGS-84)**: GPS latitude/longitude must be converted to the NED frame through an earth model to meet local navigation requirements;

- **ECEF (Earth-Centered Earth-Fixed)**: NED is a local simplified version of ECEF, suitable for small-range high-precision navigation, avoiding the computational complexity of the earth-centered frame.

In [ ]:
# Restart the AirSim environment

import threading

def plane_control(plane_name):
    # Connect to the AirSim simulator
    client = airsim.MultirotorClient()
    client.enableApiControl(True, vehicle_name=plane_name)
    client.armDisarm(True, vehicle_name=plane_name)

    client.takeoffAsync(vehicle_name=plane_name).join()
    client.moveToZAsync(-3, 1, vehicle_name=plane_name).join()  # Rise to 3m altitude, speed=1

    # Initialize 4 waypoints
    points = [airsim.Vector3r(5, 0, -3),
              airsim.Vector3r(5, 5, -3),
              airsim.Vector3r(0, 5, -3),
              airsim.Vector3r(0, 0, -3)]

    client.moveOnPathAsync(points, 1, vehicle_name=plane_name).join()

    client.landAsync(vehicle_name=plane_name).join()
    client.armDisarm(False, vehicle_name=plane_name)
    client.enableApiControl(False, vehicle_name=plane_name)


plane_name_list = []
for i in range(3):
    plane_name_list.append("UAV"+str(i+1))

In [ ]:
for plane_name in plane_name_list:
    t = threading.Thread(target=plane_control, args=(plane_name,))
    t.start()

for plane_name in plane_name_list:
    t.join()

Reference Documentation
1. https://microsoft.github.io/AirSim/multi_vehicle/
2. https://zhuanlan.zhihu.com/p/391565827
3. https://blog.csdn.net/kuvinxu/article/details/124467529
4. https://zhuanlan.zhihu.com/p/507304210